# EEG_10a — Prototypical Networks su GNN Encoder

**Task**: Few-shot meta-learning per imagined speech decoding cross-subject.
**Idea**: ogni trial EEG → embedding via GNN. Ogni classe ha un prototipo = media degli embedding del support set. Classificazione = nearest prototype (distanza euclidea).

**Setup episodico**: N=4 classi (concr4), K=5 shot, Q=10 query per classe.
**Cross-subject**: support da soggetto A, query da soggetto B → forza generalizzazione.


In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
import os

CSV_ROOT      = "/home/daniele_u/data/raw_csv/training_set"
LABEL2IDX     = "/home/daniele_u/miralis-hypergraph-imagined-speech/configs/label_schemes/label2idx.json"
LABEL2CLUSTER = "/home/daniele_u/miralis-hypergraph-imagined-speech/configs/label_schemes/labelid2cluster_concr4.json"

# Split soggetti (subject-independent)
SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

# Episodio
N_WAY      = 4     # classi concr4
K_SHOT     = 5     # support per classe
N_QUERY    = 10    # query per classe

# Grafo
K_GRAPH    = 6     # top-k vicini per PCC edge_index

# GNN
HIDDEN_DIM  = 64
EMBED_DIM   = 128
N_LAYERS    = 3
DROPOUT     = 0.3

# Training
N_EPISODES_TRAIN = 10_000
N_EPISODES_VAL   = 500
META_LR          = 1e-3
BATCH_EPISODES   = 4     # episodi per gradient step
MAX_STEPS        = 2_500

# W&B
WANDB_PROJECT = "miralis-imagined-speech"
WANDB_ENTITY  = "uras-daniele22-politecnico-di-milano"
RUN_NAME      = "eeg10a_ProtoNet_GNN_concr4"


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import json, random, glob
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch

import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# ── Carica mapping label → cluster ───────────────────────────────────────────
with open(LABEL2IDX) as f:
    label2idx = json.load(f)          # word → int (0-109)

with open(LABEL2CLUSTER) as f:
    raw = json.load(f)
    label2cluster = {int(k): int(v) for k, v in raw.items()}   # int → cluster 0-3

idx2label = {v: k for k, v in label2idx.items()}

# Raggruppa parole per cluster
cluster2words = {c: [] for c in range(N_WAY)}
for word, idx in label2idx.items():
    c = label2cluster.get(idx)
    if c is not None:
        cluster2words[c].append(word)

print("Parole per cluster:")
for c, words in cluster2words.items():
    print(f"  Cluster {c}: {len(words)} parole")


In [ ]:
# ── Trial loader ─────────────────────────────────────────────────────────────
def load_trial(csv_path: str) -> np.ndarray:
    """Carica trial EEG: shape (61, 384), float32. Normalizzazione z-score per-canale."""
    x = np.loadtxt(csv_path, delimiter=",", dtype=np.float32)
    if x.ndim == 1:
        return None
    # z-score per-canale (asse temporale)
    mu  = x.mean(axis=1, keepdims=True)
    std = x.std(axis=1, keepdims=True) + 1e-6
    return (x - mu) / std   # (61, 384)


def pcc_edge_index(x_np: np.ndarray, k: int = K_GRAPH):
    """Costruisce edge_index top-k PCC per trial. x_np: (N_CH, T)."""
    N = x_np.shape[0]
    # PCC matrix
    corr = np.corrcoef(x_np)            # (N, N)
    np.fill_diagonal(corr, -1.0)        # escludi self-loops
    # top-k vicini per nodo
    src, dst = [], []
    for i in range(N):
        topk = np.argsort(corr[i])[-k:]
        for j in topk:
            src.append(i); dst.append(j)
            src.append(j); dst.append(i)   # bidirezionale
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    return edge_index


In [ ]:
# ── SubjectTrialIndex ────────────────────────────────────────────────────────
class SubjectTrialIndex:
    """
    Per ogni soggetto nel split, indicizza i trial disponibili
    per cluster semantico. Struttura: {subj_idx: {cluster: [csv_path, ...]}}
    """
    def __init__(self, subj_indices: list):
        self.index = {}   # subj_idx → {cluster → [paths]}
        subj_dirs = sorted(glob.glob(f"{CSV_ROOT}/P*"))

        for si in subj_indices:
            if si >= len(subj_dirs):
                continue
            sdir = subj_dirs[si]
            self.index[si] = {c: [] for c in range(N_WAY)}
            for csv_path in glob.glob(f"{sdir}/**/*.csv", recursive=True):
                fname = Path(csv_path).stem          # e.g. "accendere_img"
                word  = fname.replace("_img", "").replace("_conc", "")
                idx   = label2idx.get(word)
                if idx is None:
                    continue
                cluster = label2cluster.get(idx)
                if cluster is None:
                    continue
                self.index[si][cluster].append(csv_path)

        # Rimuovi soggetti con meno di K_SHOT+N_QUERY trial per qualche cluster
        min_needed = K_SHOT + N_QUERY
        self.valid_subjects = [
            si for si, cdict in self.index.items()
            if all(len(paths) >= min_needed for paths in cdict.values())
        ]
        print(f"Soggetti validi: {len(self.valid_subjects)} / {len(subj_indices)}")

    def sample_episode(self, support_subj=None, query_subj=None):
        """
        Campiona un episodio N-way K-shot.
        support_subj e query_subj possono essere diversi (cross-subject).
        Se None, vengono scelti casualmente.
        """
        if support_subj is None:
            support_subj = random.choice(self.valid_subjects)
        if query_subj is None:
            remaining = [s for s in self.valid_subjects if s != support_subj]
            query_subj = random.choice(remaining) if remaining else support_subj

        support_x, support_y = [], []
        query_x,   query_y   = [], []

        for cluster in range(N_WAY):
            # Support
            s_paths = random.sample(self.index[support_subj][cluster], K_SHOT)
            for p in s_paths:
                x = load_trial(p)
                if x is not None:
                    support_x.append(x); support_y.append(cluster)

            # Query
            q_paths = random.sample(self.index[query_subj][cluster], N_QUERY)
            for p in q_paths:
                x = load_trial(p)
                if x is not None:
                    query_x.append(x); query_y.append(cluster)

        return support_x, support_y, query_x, query_y


# Costruisci indici
print("Building train index...")
train_index = SubjectTrialIndex(SUBJ_TRAIN)
print("Building val index...")
val_index   = SubjectTrialIndex(SUBJ_VAL)
print("Building test index...")
test_index  = SubjectTrialIndex(SUBJ_TEST)


In [ ]:
# ── Batch di grafi da lista di array numpy ───────────────────────────────────
def arrays_to_batch(x_list: list) -> Batch:
    """Converte lista di array (N_CH, T) in un Batch PyG."""
    data_list = []
    for x_np in x_list:
        x_t        = torch.from_numpy(x_np)          # (61, 384)
        edge_index = pcc_edge_index(x_np, k=K_GRAPH)
        data_list.append(Data(x=x_t, edge_index=edge_index))
    return Batch.from_data_list(data_list)


In [ ]:
# ── GNN Encoder ──────────────────────────────────────────────────────────────
class GNNEncoder(nn.Module):
    """
    GCN multi-layer con global mean pooling → embedding per trial.
    Input: nodi (N_CH, 384), edge_index → output: (batch, EMBED_DIM)
    """
    def __init__(self, in_channels=384, hidden=HIDDEN_DIM,
                 embed_dim=EMBED_DIM, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        # Prima conv
        self.convs.append(GCNConv(in_channels, hidden))
        self.bns.append(nn.BatchNorm1d(hidden))

        # Conv intermedie
        for _ in range(n_layers - 2):
            self.convs.append(GCNConv(hidden, hidden))
            self.bns.append(nn.BatchNorm1d(hidden))

        # Ultima conv → embed
        self.convs.append(GCNConv(hidden, embed_dim))
        self.bns.append(nn.BatchNorm1d(embed_dim))

        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            if i < len(self.convs) - 1:
                x = F.dropout(x, p=self.dropout, training=self.training)
        # Global pooling: aggrega i nodi di ogni grafo in un vettore
        return global_mean_pool(x, batch)   # (batch_size, embed_dim)


In [ ]:
# ── Prototypical Network ─────────────────────────────────────────────────────
class ProtoNet(nn.Module):
    """
    Wrapper che usa GNNEncoder come backbone.
    forward_episode: calcola prototipi dal support set,
    poi classifica il query set per distanza euclidea.
    """
    def __init__(self, encoder: GNNEncoder):
        super().__init__()
        self.encoder = encoder

    def compute_prototypes(self, support_batch: Batch,
                           support_labels: torch.Tensor) -> torch.Tensor:
        """
        Calcola il prototipo di ogni classe = media degli embedding.
        Ritorna: (N_WAY, EMBED_DIM)
        """
        embeddings = self.encoder(
            support_batch.x, support_batch.edge_index, support_batch.batch
        )   # (K_SHOT * N_WAY, EMBED_DIM)

        prototypes = []
        for c in range(N_WAY):
            mask = (support_labels == c)
            prototypes.append(embeddings[mask].mean(dim=0))
        return torch.stack(prototypes)   # (N_WAY, EMBED_DIM)

    def forward_episode(self, support_batch: Batch,
                         support_labels: torch.Tensor,
                         query_batch: Batch,
                         query_labels: torch.Tensor):
        """
        Ritorna loss + accuracy dell'episodio.
        """
        prototypes = self.compute_prototypes(support_batch, support_labels)

        query_emb = self.encoder(
            query_batch.x, query_batch.edge_index, query_batch.batch
        )   # (Q * N_WAY, EMBED_DIM)

        # Distanze euclidee: (n_query, N_WAY)
        dists = torch.cdist(query_emb.unsqueeze(0),
                            prototypes.unsqueeze(0)).squeeze(0)

        # Log-softmax su distanze negative
        log_p_y = F.log_softmax(-dists, dim=1)
        loss     = F.nll_loss(log_p_y, query_labels)

        # Accuracy
        preds    = log_p_y.argmax(dim=1)
        acc      = (preds == query_labels).float().mean().item()
        return loss, acc


In [ ]:
# ── Training loop ────────────────────────────────────────────────────────────
encoder = GNNEncoder().to(device)
model   = ProtoNet(encoder).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=META_LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)

run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "notebook": "EEG_10a_prototypical_gnn",
        "n_way": N_WAY, "k_shot": K_SHOT, "n_query": N_QUERY,
        "hidden_dim": HIDDEN_DIM, "embed_dim": EMBED_DIM,
        "n_layers": N_LAYERS, "dropout": DROPOUT,
        "meta_lr": META_LR, "batch_episodes": BATCH_EPISODES,
        "max_steps": MAX_STEPS, "k_graph": K_GRAPH,
        "cluster_scheme": "concr4", "model": "ProtoNet_GCN",
        "n_train_subj": len(train_index.valid_subjects),
        "n_val_subj":   len(val_index.valid_subjects),
        "n_test_subj":  len(test_index.valid_subjects),
    },
    reinit=True
)

best_val_acc = 0.0

for step in range(1, MAX_STEPS + 1):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    optimizer.zero_grad()

    for _ in range(BATCH_EPISODES):
        sx, sy, qx, qy = train_index.sample_episode()

        if len(sx) < N_WAY * K_SHOT or len(qx) < N_WAY * N_QUERY:
            continue  # episodio incompleto, salta

        s_batch = arrays_to_batch(sx).to(device)
        q_batch = arrays_to_batch(qx).to(device)
        s_labels = torch.tensor(sy, dtype=torch.long, device=device)
        q_labels = torch.tensor(qy, dtype=torch.long, device=device)

        loss, acc = model.forward_episode(s_batch, s_labels, q_batch, q_labels)
        (loss / BATCH_EPISODES).backward()
        total_loss += loss.item()
        total_acc  += acc

    optimizer.step()
    scheduler.step()

    avg_loss = total_loss / BATCH_EPISODES
    avg_acc  = total_acc  / BATCH_EPISODES

    run.log({"train/loss": avg_loss, "train/acc": avg_acc,
             "lr": scheduler.get_last_lr()[0], "step": step})

    # ── Validation ogni 100 step ──
    if step % 100 == 0:
        model.eval()
        val_accs = []
        with torch.no_grad():
            for _ in range(N_EPISODES_VAL // 10):   # 50 episodi veloci
                sx, sy, qx, qy = val_index.sample_episode()
                if len(sx) < N_WAY * K_SHOT or len(qx) < N_WAY * N_QUERY:
                    continue
                s_b = arrays_to_batch(sx).to(device)
                q_b = arrays_to_batch(qx).to(device)
                _, acc = model.forward_episode(
                    s_b, torch.tensor(sy, dtype=torch.long, device=device),
                    q_b, torch.tensor(qy, dtype=torch.long, device=device)
                )
                val_accs.append(acc)
        val_acc = float(np.mean(val_accs)) if val_accs else 0.0
        run.log({"val/acc": val_acc, "step": step})
        print(f"Step {step:4d} | loss {avg_loss:.4f} | train_acc {avg_acc:.3f} | val_acc {val_acc:.3f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "/tmp/protonet_best.pt")
            run.summary["best_val_acc"] = best_val_acc

print(f"\nMigliore val_acc: {best_val_acc:.3f}")


In [ ]:
# ── Test finale ──────────────────────────────────────────────────────────────
model.load_state_dict(torch.load("/tmp/protonet_best.pt"))
model.eval()

test_accs = []
with torch.no_grad():
    for _ in range(N_EPISODES_VAL):
        sx, sy, qx, qy = test_index.sample_episode()
        if len(sx) < N_WAY * K_SHOT or len(qx) < N_WAY * N_QUERY:
            continue
        s_b = arrays_to_batch(sx).to(device)
        q_b = arrays_to_batch(qx).to(device)
        _, acc = model.forward_episode(
            s_b, torch.tensor(sy, dtype=torch.long, device=device),
            q_b, torch.tensor(qy, dtype=torch.long, device=device)
        )
        test_accs.append(acc)

test_acc = float(np.mean(test_accs))
test_ci  = 1.96 * float(np.std(test_accs)) / np.sqrt(len(test_accs))

print(f"Test acc (N={len(test_accs)} ep): {test_acc:.3f} ± {test_ci:.3f}")
print(f"Chance level: {1/N_WAY:.3f}")

run.summary["test_acc"]    = test_acc
run.summary["test_ci_95"]  = test_ci
run.summary["n_test_ep"]   = len(test_accs)
run.finish()
